# U.S. Airline Operational Performance Analytics 2025
## 03 — Data Enrichment & Database Export


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print(f"Pandas version: {pd.__version__}")
print("Libraries imported successfully.")


Pandas version: 3.0.3
Libraries imported successfully.


## 2. Detect Project Folder

In [2]:
current_path = Path.cwd().resolve()

if (current_path / "data" / "processed").exists():
    project_root = current_path
elif (current_path.parent / "data" / "processed").exists():
    project_root = current_path.parent
else:
    raise FileNotFoundError(
        "Folder data/processed tidak ditemukan. Pastikan notebook berada di project yang benar."
    )

processed_path = project_root / "data" / "processed"
analysis_path = project_root / "data" / "analysis"
database_path = project_root / "data" / "database"

database_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Processed path: {processed_path}")
print(f"Analysis path: {analysis_path}")
print(f"Database export path: {database_path}")


Project root: C:\Bootcamp Dibimbing\Assignment\D30_Portfolio
Processed path: C:\Bootcamp Dibimbing\Assignment\D30_Portfolio\data\processed
Analysis path: C:\Bootcamp Dibimbing\Assignment\D30_Portfolio\data\analysis
Database export path: C:\Bootcamp Dibimbing\Assignment\D30_Portfolio\data\database


## 3. Validate Input Files

In [3]:
processed_files = sorted(
    processed_path.glob("airline_2025_*_processed.csv.gz")
)

print(f"Processed monthly files found: {len(processed_files)}")

if len(processed_files) != 12:
    raise ValueError(
        f"Expected 12 processed files, but found {len(processed_files)}."
    )

route_analysis_file = analysis_path / "route_performance_2025.csv"

if not route_analysis_file.exists():
    raise FileNotFoundError(
        "route_performance_2025.csv tidak ditemukan. Jalankan notebook 02 EDA terlebih dahulu."
    )

print("Input validation passed.")


Processed monthly files found: 12
Input validation passed.


## 4. Airline Name Mapping

Dataset BTS menggunakan carrier code seperti `AA`, `DL`, `F9`, dan `WN`.

Untuk dashboard dan portfolio, nama tersebut akan dibuat lebih mudah dibaca.


In [4]:
airline_name_map = {
    "AA": "American Airlines",
    "AS": "Alaska Airlines",
    "B6": "JetBlue Airways",
    "DL": "Delta Air Lines",
    "F9": "Frontier Airlines",
    "G4": "Allegiant Air",
    "HA": "Hawaiian Airlines",
    "MQ": "Envoy Air",
    "NK": "Spirit Airlines",
    "OH": "PSA Airlines",
    "OO": "SkyWest Airlines",
    "UA": "United Airlines",
    "WN": "Southwest Airlines",
    "YX": "Republic Airways"
}

print(f"Airline mappings prepared: {len(airline_name_map)}")


Airline mappings prepared: 14


## 5. Build `dim_airline`

In [5]:
airline_parts = []

for file in processed_files:
    airline_part = pd.read_csv(
        file,
        usecols=[
            "DOT_ID_Reporting_Airline",
            "Reporting_Airline"
        ],
        compression="gzip"
    ).drop_duplicates()

    airline_parts.append(airline_part)

dim_airline = (
    pd.concat(airline_parts, ignore_index=True)
    .drop_duplicates()
    .sort_values("Reporting_Airline")
    .reset_index(drop=True)
)

dim_airline["AirlineName"] = (
    dim_airline["Reporting_Airline"]
    .map(airline_name_map)
)

unmapped_airlines = dim_airline[
    dim_airline["AirlineName"].isna()
]

if len(unmapped_airlines) > 0:
    raise ValueError(
        f"Unmapped airline codes found: "
        f"{unmapped_airlines['Reporting_Airline'].tolist()}"
    )

dim_airline = dim_airline.rename(
    columns={
        "DOT_ID_Reporting_Airline": "AirlineKey",
        "Reporting_Airline": "AirlineCode"
    }
)

print(f"Airlines in dimension: {len(dim_airline)}")

dim_airline


Airlines in dimension: 14


,AirlineCode,AirlineKey,AirlineName
0,AA,19805,American Airlines
1,AS,19930,Alaska Airlines
2,B6,20409,JetBlue Airways
3,DL,19790,Delta Air Lines
4,F9,20436,Frontier Airlines
5,G4,20368,Allegiant Air
6,HA,19690,Hawaiian Airlines
7,MQ,20398,Envoy Air
8,NK,20416,Spirit Airlines
9,OH,20397,PSA Airlines


In [6]:
dim_airline.to_csv(
    database_path / "dim_airline.csv",
    index=False
)

print("dim_airline.csv saved.")


dim_airline.csv saved.


## 6. Build `dim_airport`

In [7]:
airport_parts = []

for file_number, file in enumerate(processed_files, start=1):
    print(f"Reading airport data {file_number}/12: {file.name}")

    origin_df = pd.read_csv(
        file,
        usecols=[
            "OriginAirportID",
            "Origin",
            "OriginCityName",
            "OriginState",
            "OriginStateName"
        ],
        compression="gzip"
    )

    origin_df = origin_df.rename(
        columns={
            "OriginAirportID": "AirportKey",
            "Origin": "AirportCode",
            "OriginCityName": "CityName",
            "OriginState": "StateCode",
            "OriginStateName": "StateName"
        }
    )

    destination_df = pd.read_csv(
        file,
        usecols=[
            "DestAirportID",
            "Dest",
            "DestCityName",
            "DestState",
            "DestStateName"
        ],
        compression="gzip"
    )

    destination_df = destination_df.rename(
        columns={
            "DestAirportID": "AirportKey",
            "Dest": "AirportCode",
            "DestCityName": "CityName",
            "DestState": "StateCode",
            "DestStateName": "StateName"
        }
    )

    airport_parts.append(
        origin_df.drop_duplicates()
    )

    airport_parts.append(
        destination_df.drop_duplicates()
    )

dim_airport = (
    pd.concat(airport_parts, ignore_index=True)
    .drop_duplicates()
    .sort_values("AirportKey")
    .reset_index(drop=True)
)

print(f"Airport dimension rows before validation: {len(dim_airport):,}")


Reading airport data 1/12: airline_2025_01_processed.csv.gz
Reading airport data 2/12: airline_2025_02_processed.csv.gz
Reading airport data 3/12: airline_2025_03_processed.csv.gz
Reading airport data 4/12: airline_2025_04_processed.csv.gz
Reading airport data 5/12: airline_2025_05_processed.csv.gz
Reading airport data 6/12: airline_2025_06_processed.csv.gz
Reading airport data 7/12: airline_2025_07_processed.csv.gz
Reading airport data 8/12: airline_2025_08_processed.csv.gz
Reading airport data 9/12: airline_2025_09_processed.csv.gz
Reading airport data 10/12: airline_2025_10_processed.csv.gz
Reading airport data 11/12: airline_2025_11_processed.csv.gz
Reading airport data 12/12: airline_2025_12_processed.csv.gz
Airport dimension rows before validation: 352


In [8]:
airport_key_check = (
    dim_airport.groupby("AirportKey")["AirportCode"]
    .nunique()
)

conflicting_airport_keys = airport_key_check[
    airport_key_check > 1
]

if len(conflicting_airport_keys) > 0:
    raise ValueError(
        f"Conflicting airport codes found for AirportKey: "
        f"{conflicting_airport_keys.index.tolist()}"
    )

dim_airport = (
    dim_airport
    .drop_duplicates(subset=["AirportKey"])
    .copy()
)

dim_airport["AirportLabel"] = (
    dim_airport["AirportCode"]
    + " - "
    + dim_airport["CityName"]
)

print(f"Unique airports: {len(dim_airport):,}")

dim_airport.head(10)


Unique airports: 352


,AirportKey,AirportCode,CityName,StateCode,StateName,AirportLabel
0,10135,ABE,"Allentown/Bethlehem/Easton, PA",PA,Pennsylvania,"ABE - Allentown/Bethlehem/Easton, PA"
1,10136,ABI,"Abilene, TX",TX,Texas,"ABI - Abilene, TX"
2,10140,ABQ,"Albuquerque, NM",NM,New Mexico,"ABQ - Albuquerque, NM"
3,10141,ABR,"Aberdeen, SD",SD,South Dakota,"ABR - Aberdeen, SD"
4,10146,ABY,"Albany, GA",GA,Georgia,"ABY - Albany, GA"
5,10154,ACK,"Nantucket, MA",MA,Massachusetts,"ACK - Nantucket, MA"
6,10155,ACT,"Waco, TX",TX,Texas,"ACT - Waco, TX"
7,10157,ACV,"Arcata/Eureka, CA",CA,California,"ACV - Arcata/Eureka, CA"
8,10158,ACY,"Atlantic City, NJ",NJ,New Jersey,"ACY - Atlantic City, NJ"
9,10165,ADK,"Adak Island, AK",AK,Alaska,"ADK - Adak Island, AK"


In [9]:
dim_airport.to_csv(
    database_path / "dim_airport.csv",
    index=False
)

print("dim_airport.csv saved.")


dim_airport.csv saved.


## 7. Build `dim_date`

Date dimension dibuat untuk seluruh tanggal selama tahun 2025.

Ini akan memudahkan analisis:
- year,
- quarter,
- month,
- day of week,
- weekday vs weekend.


In [10]:
date_range = pd.date_range(
    start="2025-01-01",
    end="2025-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "Date": date_range
})

dim_date["DateKey"] = (
    dim_date["Date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = dim_date["Date"].dt.quarter
dim_date["MonthNumber"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()
dim_date["DayOfMonth"] = dim_date["Date"].dt.day
dim_date["DayOfWeekNumber"] = dim_date["Date"].dt.dayofweek + 1
dim_date["DayName"] = dim_date["Date"].dt.day_name()
dim_date["WeekendFlag"] = (
    dim_date["Date"].dt.dayofweek >= 5
).astype(int)

dim_date = dim_date[
    [
        "DateKey",
        "Date",
        "Year",
        "Quarter",
        "MonthNumber",
        "MonthName",
        "DayOfMonth",
        "DayOfWeekNumber",
        "DayName",
        "WeekendFlag"
    ]
]

print(f"Dates in dimension: {len(dim_date)}")

dim_date.head()


Dates in dimension: 365


,DateKey,Date,Year,Quarter,MonthNumber,MonthName,DayOfMonth,DayOfWeekNumber,DayName,WeekendFlag
0,20250101,2025-01-01,2025,1,1,January,1,3,Wednesday,0
1,20250102,2025-01-02,2025,1,1,January,2,4,Thursday,0
2,20250103,2025-01-03,2025,1,1,January,3,5,Friday,0
3,20250104,2025-01-04,2025,1,1,January,4,6,Saturday,1
4,20250105,2025-01-05,2025,1,1,January,5,7,Sunday,1


In [11]:
dim_date.to_csv(
    database_path / "dim_date.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("dim_date.csv saved.")


dim_date.csv saved.


## 8. Route Priority Analysis

EDA sebelumnya menunjukkan route dengan **delay rate tertinggi**.

Tetapi business priority tidak sebaiknya hanya melihat persentase.

Contoh:

```text
Route A
1,000 flights
40% delayed

Route B
10,000 flights
30% delayed
```

Walaupun Route A memiliki delay rate lebih tinggi, Route B bisa memberikan operational impact jauh lebih besar.

Karena itu prioritas route akan mempertimbangkan:

> **Total delayed arrival minutes**

Metric ini mencerminkan kombinasi antara:
- berapa banyak flight yang delayed,
- seberapa berat delay tersebut.


In [12]:
route_df = pd.read_csv(
    route_analysis_file
)

route_priority_df = route_df[
    route_df["TotalFlights"] >= 1_000
].copy()

route_priority_df["DelayBurdenHours"] = (
    route_priority_df["DelayedArrivalMinutes"] / 60
)

route_priority_df = route_priority_df.sort_values(
    "DelayedArrivalMinutes",
    ascending=False
).reset_index(drop=True)

route_priority_df["PriorityRank"] = (
    route_priority_df.index + 1
)

route_priority_df = route_priority_df[
    [
        "PriorityRank",
        "Origin",
        "Dest",
        "Route",
        "TotalFlights",
        "ValidArrivals",
        "Delayed15Flights",
        "ArrivalDelayRatePct",
        "SevereDelayRatePct",
        "AvgDelayMinutesDelayedFlights",
        "DelayedArrivalMinutes",
        "DelayBurdenHours"
    ]
]

print("Top 15 routes by total delay burden:")

route_priority_df.head(15)


Top 15 routes by total delay burden:


,PriorityRank,Origin,Dest,Route,TotalFlights,ValidArrivals,Delayed15Flights,ArrivalDelayRatePct,SevereDelayRatePct,AvgDelayMinutesDelayedFlights,DelayedArrivalMinutes,DelayBurdenHours
0,1,LGA,ORD,LGA - ORD,11545,11206,3379,30.153489,13.698019,88.501036,299045.0,4984.083333
1,2,ORD,LGA,ORD - LGA,11542,11176,3327,29.769148,13.958482,85.897806,285782.0,4763.033333
2,3,MCO,EWR,MCO - EWR,6582,6416,1923,29.971945,15.243142,98.897556,190180.0,3169.666667
3,4,ATL,MCO,ATL - MCO,8143,8022,2560,31.912241,12.528048,74.271094,190134.0,3168.900000
4,5,DCA,BOS,DCA - BOS,9131,8829,2391,27.081210,12.390984,78.040569,186595.0,3109.916667
5,6,ATL,EWR,ATL - EWR,6105,5910,2073,35.076142,16.835871,86.553787,179426.0,2990.433333
6,7,ORD,EWR,ORD - EWR,5990,5838,1976,33.847208,16.752312,89.873988,177591.0,2959.850000
7,8,MCO,ATL,MCO - ATL,8140,8036,2116,26.331508,11.050274,82.229679,173998.0,2899.966667
8,9,LAX,JFK,LAX - JFK,10265,10110,1935,19.139466,8.407517,89.431525,173050.0,2884.166667
9,10,BOS,DCA,BOS - DCA,9132,8813,2341,26.563032,10.654715,73.465186,171982.0,2866.366667


In [13]:
route_priority_df.to_csv(
    database_path / "route_priority_2025.csv",
    index=False
)

print("route_priority_2025.csv saved.")


route_priority_2025.csv saved.


## 9. Define Final Fact Table Columns

`fact_flights` memiliki grain:

> **1 row = 1 scheduled flight record**

Dimension keys:
- `DateKey`
- `AirlineKey`
- `OriginAirportKey`
- `DestinationAirportKey`

Business metrics tetap berada di fact table.


In [14]:
fact_source_columns = [
    "FlightDate",
    "DOT_ID_Reporting_Airline",
    "OriginAirportID",
    "DestAirportID",
    "Flight_Number_Reporting_Airline",
    "Tail_Number",
    "Route",

    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "DepTimeBlk",
    "TaxiOut",

    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrTimeBlk",
    "TaxiIn",

    "Cancelled",
    "CancellationCode",
    "Diverted",

    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",

    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",

    "ScheduledDepHour",
    "DeparturePeriod",
    "ValidArrivalFlag",
    "ArrivalDelayed15Flag",
    "OnTimeArrivalFlag",
    "SevereDelayFlag",
    "ArrivalStatus",
    "DelaySeverity",
    "TotalCauseDelayMinutes",
    "NetDelayChangeMinutes",
    "DistanceBand"
]

print(f"Fact source columns: {len(fact_source_columns)}")


Fact source columns: 44


## 10. Export Final Monthly Fact Tables

Fact table tetap dibagi per bulan supaya:
- file lebih manageable,
- loading PostgreSQL nanti lebih aman,
- tidak perlu membuat satu file raksasa.

Total row keseluruhan tetap sama dengan processed dataset.


In [15]:
export_audit = []

valid_airline_keys = set(
    dim_airline["AirlineKey"].dropna().astype(int)
)

valid_airport_keys = set(
    dim_airport["AirportKey"].dropna().astype(int)
)

valid_date_keys = set(
    dim_date["DateKey"].astype(int)
)

for month_number, file in enumerate(processed_files, start=1):
    print("")
    print("=" * 70)
    print(f"Preparing database fact file {month_number}/12")
    print(f"Source: {file.name}")

    df = pd.read_csv(
        file,
        usecols=fact_source_columns,
        compression="gzip",
        low_memory=False
    )

    rows_source = len(df)

    df["FlightDate"] = pd.to_datetime(
        df["FlightDate"],
        errors="coerce"
    )

    df["DateKey"] = (
        df["FlightDate"]
        .dt.strftime("%Y%m%d")
        .astype("Int64")
    )

    df = df.rename(
        columns={
            "DOT_ID_Reporting_Airline": "AirlineKey",
            "OriginAirportID": "OriginAirportKey",
            "DestAirportID": "DestinationAirportKey",
            "Flight_Number_Reporting_Airline": "FlightNumber"
        }
    )

    # Foreign key validation
    invalid_airline = (
        ~df["AirlineKey"]
        .dropna()
        .astype(int)
        .isin(valid_airline_keys)
    ).sum()

    invalid_origin = (
        ~df["OriginAirportKey"]
        .dropna()
        .astype(int)
        .isin(valid_airport_keys)
    ).sum()

    invalid_destination = (
        ~df["DestinationAirportKey"]
        .dropna()
        .astype(int)
        .isin(valid_airport_keys)
    ).sum()

    invalid_date = (
        ~df["DateKey"]
        .dropna()
        .astype(int)
        .isin(valid_date_keys)
    ).sum()

    if (
        invalid_airline > 0
        or invalid_origin > 0
        or invalid_destination > 0
        or invalid_date > 0
    ):
        raise ValueError(
            f"Foreign key validation failed for month {month_number:02d}."
        )

    fact_columns = [
        "DateKey",
        "AirlineKey",
        "OriginAirportKey",
        "DestinationAirportKey",
        "FlightNumber",
        "Tail_Number",
        "Route",

        "CRSDepTime",
        "DepTime",
        "DepDelay",
        "DepDelayMinutes",
        "DepDel15",
        "DepTimeBlk",
        "TaxiOut",

        "CRSArrTime",
        "ArrTime",
        "ArrDelay",
        "ArrDelayMinutes",
        "ArrDel15",
        "ArrTimeBlk",
        "TaxiIn",

        "Cancelled",
        "CancellationCode",
        "Diverted",

        "CRSElapsedTime",
        "ActualElapsedTime",
        "AirTime",
        "Distance",

        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay",

        "ScheduledDepHour",
        "DeparturePeriod",
        "ValidArrivalFlag",
        "ArrivalDelayed15Flag",
        "OnTimeArrivalFlag",
        "SevereDelayFlag",
        "ArrivalStatus",
        "DelaySeverity",
        "TotalCauseDelayMinutes",
        "NetDelayChangeMinutes",
        "DistanceBand"
    ]

    fact_df = df[fact_columns].copy()

    output_file = (
        database_path
        / f"fact_flights_2025_{month_number:02d}.csv.gz"
    )

    fact_df.to_csv(
        output_file,
        index=False,
        compression={
            "method": "gzip",
            "compresslevel": 3
        }
    )

    rows_exported = len(fact_df)

    export_audit.append({
        "Month": month_number,
        "SourceRows": rows_source,
        "ExportedRows": rows_exported,
        "InvalidAirlineKeys": int(invalid_airline),
        "InvalidOriginAirportKeys": int(invalid_origin),
        "InvalidDestinationAirportKeys": int(invalid_destination),
        "InvalidDateKeys": int(invalid_date)
    })

    print(f"Rows exported: {rows_exported:,}")
    print(f"Output: {output_file.name}")

    del df
    del fact_df

print("")
print("All monthly fact files exported successfully.")



Preparing database fact file 1/12
Source: airline_2025_01_processed.csv.gz
Rows exported: 539,747
Output: fact_flights_2025_01.csv.gz

Preparing database fact file 2/12
Source: airline_2025_02_processed.csv.gz
Rows exported: 504,884
Output: fact_flights_2025_02.csv.gz

Preparing database fact file 3/12
Source: airline_2025_03_processed.csv.gz
Rows exported: 600,872
Output: fact_flights_2025_03.csv.gz

Preparing database fact file 4/12
Source: airline_2025_04_processed.csv.gz
Rows exported: 583,950
Output: fact_flights_2025_04.csv.gz

Preparing database fact file 5/12
Source: airline_2025_05_processed.csv.gz
Rows exported: 605,648
Output: fact_flights_2025_05.csv.gz

Preparing database fact file 6/12
Source: airline_2025_06_processed.csv.gz
Rows exported: 611,575
Output: fact_flights_2025_06.csv.gz

Preparing database fact file 7/12
Source: airline_2025_07_processed.csv.gz
Rows exported: 631,428
Output: fact_flights_2025_07.csv.gz

Preparing database fact file 8/12
Source: airline_2025

## 11. Database Export Audit

In [16]:
export_audit_df = pd.DataFrame(
    export_audit
)

export_audit_df


,Month,SourceRows,ExportedRows,InvalidAirlineKeys,InvalidOriginAirportKeys,InvalidDestinationAirportKeys,InvalidDateKeys
0,1,539747,539747,0,0,0,0
1,2,504884,504884,0,0,0,0
2,3,600872,600872,0,0,0,0
3,4,583950,583950,0,0,0,0
4,5,605648,605648,0,0,0,0
5,6,611575,611575,0,0,0,0
6,7,631428,631428,0,0,0,0
7,8,602378,602378,0,0,0,0
8,9,562439,562439,0,0,0,0
9,10,605844,605844,0,0,0,0


In [17]:
export_audit_df.to_csv(
    database_path / "database_export_audit.csv",
    index=False
)

total_source_rows = int(
    export_audit_df["SourceRows"].sum()
)

total_exported_rows = int(
    export_audit_df["ExportedRows"].sum()
)

print(f"Total source rows: {total_source_rows:,}")
print(f"Total exported fact rows: {total_exported_rows:,}")

if total_source_rows != total_exported_rows:
    raise ValueError(
        "Total source rows and exported rows do not match."
    )

print("Row-count validation passed.")


Total source rows: 7,001,619
Total exported fact rows: 7,001,619
Row-count validation passed.


## 12. Validate Database Export Files

In [18]:
dimension_files = [
    database_path / "dim_airline.csv",
    database_path / "dim_airport.csv",
    database_path / "dim_date.csv"
]

fact_files = sorted(
    database_path.glob("fact_flights_2025_*.csv.gz")
)

print(f"Dimension files: {len(dimension_files)}")
print(f"Fact monthly files: {len(fact_files)}")
print("")

for file in dimension_files:
    print(f"{file.name} | {file.stat().st_size / 1024:.2f} KB")

print("")

for file in fact_files:
    print(
        f"{file.name} | "
        f"{file.stat().st_size / (1024 ** 2):.2f} MB"
    )

if len(fact_files) != 12:
    raise ValueError(
        f"Expected 12 fact files, but found {len(fact_files)}."
    )


Dimension files: 3
Fact monthly files: 12

dim_airline.csv | 0.39 KB
dim_airport.csv | 21.84 KB
dim_date.csv | 18.72 KB

fact_flights_2025_01.csv.gz | 25.11 MB
fact_flights_2025_02.csv.gz | 23.52 MB
fact_flights_2025_03.csv.gz | 28.26 MB
fact_flights_2025_04.csv.gz | 27.20 MB
fact_flights_2025_05.csv.gz | 28.54 MB
fact_flights_2025_06.csv.gz | 28.85 MB
fact_flights_2025_07.csv.gz | 29.80 MB
fact_flights_2025_08.csv.gz | 28.51 MB
fact_flights_2025_09.csv.gz | 25.95 MB
fact_flights_2025_10.csv.gz | 28.40 MB
fact_flights_2025_11.csv.gz | 26.67 MB
fact_flights_2025_12.csv.gz | 28.12 MB


## 13. Final Validation Summary

In [19]:
print("DATABASE EXPORT SUMMARY")
print("=" * 70)

print(f"Airlines: {len(dim_airline):,}")
print(f"Airports: {len(dim_airport):,}")
print(f"Dates: {len(dim_date):,}")
print(f"Fact rows: {total_exported_rows:,}")
print(f"Monthly fact files: {len(fact_files)}")

print("")
print("Top 10 routes by delay burden:")

display(
    route_priority_df.head(10)
)


DATABASE EXPORT SUMMARY
Airlines: 14
Airports: 352
Dates: 365
Fact rows: 7,001,619
Monthly fact files: 12

Top 10 routes by delay burden:


,PriorityRank,Origin,Dest,Route,TotalFlights,ValidArrivals,Delayed15Flights,ArrivalDelayRatePct,SevereDelayRatePct,AvgDelayMinutesDelayedFlights,DelayedArrivalMinutes,DelayBurdenHours
0,1,LGA,ORD,LGA - ORD,11545,11206,3379,30.153489,13.698019,88.501036,299045.0,4984.083333
1,2,ORD,LGA,ORD - LGA,11542,11176,3327,29.769148,13.958482,85.897806,285782.0,4763.033333
2,3,MCO,EWR,MCO - EWR,6582,6416,1923,29.971945,15.243142,98.897556,190180.0,3169.666667
3,4,ATL,MCO,ATL - MCO,8143,8022,2560,31.912241,12.528048,74.271094,190134.0,3168.900000
4,5,DCA,BOS,DCA - BOS,9131,8829,2391,27.081210,12.390984,78.040569,186595.0,3109.916667
5,6,ATL,EWR,ATL - EWR,6105,5910,2073,35.076142,16.835871,86.553787,179426.0,2990.433333
6,7,ORD,EWR,ORD - EWR,5990,5838,1976,33.847208,16.752312,89.873988,177591.0,2959.850000
7,8,MCO,ATL,MCO - ATL,8140,8036,2116,26.331508,11.050274,82.229679,173998.0,2899.966667
8,9,LAX,JFK,LAX - JFK,10265,10110,1935,19.139466,8.407517,89.431525,173050.0,2884.166667
9,10,BOS,DCA,BOS - DCA,9132,8813,2341,26.563032,10.654715,73.465186,171982.0,2866.366667
